# MY FARM — Beans Disease Model Training

Trains the beans classifier on the **ibean** dataset (Makerere AI Lab / NaCRRI) — real Ugandan field photos of healthy, angular leaf spot and bean rust leaves.

This version downloads the data from **Hugging Face** (reliable mirror of the official Makerere dataset). No Kaggle, no manual upload.

### How to run
1. **Runtime → Change runtime type → GPU**.
2. **Runtime → Run all**.
3. Download `beans.tflite` and `beans_labels.txt` at the end.


## 1. GPU check

In [ ]:
import tensorflow as tf
print('TensorFlow', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'NO GPU — Runtime > Change runtime type > GPU')

## 2. Download the beans dataset from Hugging Face
The `datasets` library is preinstalled in Colab. This pulls the official Makerere ibean dataset and saves the images into class folders on disk.

In [ ]:
from datasets import load_dataset
import os

print('Downloading ibean dataset from Hugging Face...')
ds = load_dataset('AI-Lab-Makerere/beans')   # has 'train','validation','test'
print(ds)

# HF label ids: 0 = angular_leaf_spot, 1 = bean_rust, 2 = healthy
hf_label_names = ds['train'].features['labels'].names
print('HF classes:', hf_label_names)

# Save images to folders: /content/beans_data/<split>/<classname>/img.jpg
BASE = '/content/beans_data'
for split in ['train', 'validation']:
    for i, ex in enumerate(ds[split]):
        cls = hf_label_names[ex['labels']]
        d = f'{BASE}/{split}/{cls}'
        os.makedirs(d, exist_ok=True)
        ex['image'].convert('RGB').save(f'{d}/{i}.jpg')
    print(f'Saved {split}:', sum(len(files) for _,_,files in os.walk(f'{BASE}/{split}')), 'images')
print('Done saving images.')

## 3. Build the data pipeline
Load the saved folders. Keras sorts class folders alphabetically: angular_leaf_spot, bean_rust, healthy — we map those to the app's treatment keys.

In [ ]:
import numpy as np
IMG_SIZE = 224
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/beans_data/train', image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, shuffle=True)
val_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/beans_data/validation', image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, shuffle=False)

folder_names = train_ds.class_names
print('Folder order:', folder_names)

APP_KEY = {
    'angular_leaf_spot': 'bean_angular_leaf_spot',
    'bean_rust': 'bean_rust',
    'healthy': 'healthy',
}
class_names = [APP_KEY[f] for f in folder_names]
print('Label keys (match treatment_db.dart):', class_names)

# class weights for imbalance
from sklearn.utils.class_weight import compute_class_weight
y = np.concatenate([y.numpy() for _, y in train_ds], axis=0)
w = compute_class_weight('balanced', classes=np.arange(len(class_names)), y=y)
class_weight = {i: float(x) for i, x in enumerate(w)}
print('Class weights:', class_weight)

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

## 4. Build the model (transfer learning)

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

data_aug = models.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
], name='augment')

base = EfficientNetB0(include_top=False, weights='imagenet',
                      input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False

inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3))
x = data_aug(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)
model = models.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 5. Phase 1 — train the head

In [ ]:
cbs1 = [tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor='val_accuracy'),
        tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.3, min_lr=1e-5)]
model.fit(train_ds, validation_data=val_ds, epochs=15, class_weight=class_weight, callbacks=cbs1)

## 6. Phase 2 — fine-tune

In [ ]:
base.trainable = True
for layer in base.layers[:-60]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cbs2 = [tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor='val_accuracy'),
        tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.3, min_lr=1e-6)]
model.fit(train_ds, validation_data=val_ds, epochs=15, class_weight=class_weight, callbacks=cbs2)

## 7. Evaluate

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
y_true, y_pred = [], []
for imgs, labels in val_ds:
    p = model.predict(imgs, verbose=0)
    y_pred.extend(np.argmax(p, axis=1)); y_true.extend(labels.numpy())
print(classification_report(y_true, y_pred, target_names=class_names))
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5,4)); ax.imshow(cm, cmap='Greens')
ax.set_xticks(range(len(class_names))); ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha='right'); ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title('Confusion Matrix')
for i in range(len(class_names)):
    for j in range(len(class_names)):
        ax.text(j,i,cm[i,j],ha='center',color='white' if cm[i,j]>cm.max()/2 else 'black')
plt.tight_layout(); plt.savefig('beans_confusion_matrix.png', dpi=120); plt.show()

## 8. Export to TensorFlow Lite

In [ ]:
conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
tflite = conv.convert()
open('beans.tflite','wb').write(tflite)
open('beans_labels.txt','w').write('\n'.join(class_names))
print('beans.tflite', round(len(tflite)/1e6,1), 'MB'); print('Labels:', class_names)

## 9. Download for the app

In [ ]:
from google.colab import files
files.download('beans.tflite')
files.download('beans_labels.txt')
files.download('beans_confusion_matrix.png')
print('Put beans.tflite and beans_labels.txt into: myfarm_flutter/assets/models/')